# Akselerasi GPU NVIDIA untuk Deep Learning

**Kursus NCA-GENL — Navasena | Modul 02: Deep Learning Fundamentals**

---

## Tujuan Pembelajaran (Learning Objectives)

Setelah menyelesaikan notebook ini, kamu akan mampu:

1. Memahami **mengapa GPU diperlukan** untuk deep learning (parallel processing)
2. Melakukan **benchmark CPU vs GPU** pada operasi matriks, training, dan inference
3. Mengenal **ekosistem NVIDIA** untuk deep learning (CUDA, cuDNN, TensorRT)
4. Menggunakan **Mixed Precision Training** untuk mempercepat pelatihan model
5. Memahami konsep dasar **TensorRT** untuk optimasi inference

**Estimasi waktu:** ~60 menit

> **Prasyarat:** Pastikan kamu sudah mengaktifkan GPU di Colab: **Runtime → Change runtime type → T4 GPU**

In [ ]:
!pip install -q tensorflow matplotlib numpy

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import time

print(f"TensorFlow version: {tf.__version__}")
print(f"Built with CUDA  : {tf.test.is_built_with_cuda()}")
print()

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU terdeteksi: {len(gpus)} device")
    for gpu in gpus:
        print(f"  {gpu}")
    # Show GPU details
    !nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
else:
    print("Tidak ada GPU terdeteksi.")
    print("Aktifkan GPU: Runtime -> Change runtime type -> T4 GPU")

## Bagian 1: Mengapa Deep Learning Butuh GPU?

### CPU vs GPU — Analogi

Bayangkan kamu harus menghitung nilai ujian 1.000 siswa:
- **CPU** = Satu guru jenius yang menghitung satu per satu. Sangat pintar, tapi sendirian. → **Serial processing**
- **GPU** = 1.000 kalkulator sederhana yang bekerja bersamaan. Masing-masing hanya bisa hitung sederhana, tapi serentak! → **Parallel processing**

Deep learning = jutaan operasi perkalian matriks (matrix multiplication) yang BISA dilakukan bersamaan. Inilah mengapa GPU NVIDIA jauh lebih cepat!

| Aspek | CPU | GPU NVIDIA |
|-------|-----|------------|
| Core | 4–16 core (kuat) | 2.560–16.384 core (sederhana) |
| Cocok untuk | Tugas kompleks berurutan | Tugas sederhana paralel |
| Deep Learning | Lambat (jam–hari) | Cepat (menit–jam) |
| Contoh | Intel i7, AMD Ryzen | T4, A100, H100 |

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# CPU: few large cores
cpu_cores = 8
for i in range(cpu_cores):
    row, col = divmod(i, 4)
    rect = plt.Rectangle((col*1.2, row*1.2), 1, 1, facecolor='#42A5F5', edgecolor='white', linewidth=2)
    ax1.add_patch(rect)
    ax1.text(col*1.2 + 0.5, row*1.2 + 0.5, f'Core\n{i+1}', ha='center', va='center', fontsize=9, fontweight='bold')
ax1.set_xlim(-0.2, 5)
ax1.set_ylim(-0.2, 2.8)
ax1.set_title(f'CPU: {cpu_cores} Core Besar (Kuat tapi Sedikit)', fontsize=13, fontweight='bold')
ax1.set_aspect('equal')
ax1.axis('off')

# GPU: many small cores
gpu_rows, gpu_cols = 10, 20
for r in range(gpu_rows):
    for c in range(gpu_cols):
        rect = plt.Rectangle((c*0.24, r*0.24), 0.2, 0.2, facecolor='#76B900', edgecolor='white', linewidth=0.5)
        ax2.add_patch(rect)
ax2.set_xlim(-0.1, gpu_cols*0.24 + 0.1)
ax2.set_ylim(-0.1, gpu_rows*0.24 + 0.1)
ax2.set_title(f'GPU: {gpu_rows*gpu_cols}+ Core Kecil (Sederhana tapi Banyak!)', fontsize=13, fontweight='bold')
ax2.set_aspect('equal')
ax2.axis('off')

plt.suptitle('Mengapa GPU Lebih Cepat untuk Deep Learning?', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## Bagian 2: Benchmark CPU vs GPU

### Operasi Dasar: Perkalian Matriks (Matrix Multiplication)

Perkalian matriks adalah operasi inti (core operation) deep learning. Setiap layer neural network = perkalian matriks. Mari kita ukur perbedaan kecepatan.

In [ ]:
def benchmark_matmul(size, device, n_runs=5):
    """Benchmark perkalian matriks pada device tertentu"""
    with tf.device(device):
        a = tf.random.normal([size, size])
        b = tf.random.normal([size, size])
        # Warmup
        _ = tf.matmul(a, b).numpy()

        times = []
        for _ in range(n_runs):
            start = time.time()
            c = tf.matmul(a, b)
            _ = c.numpy()  # force execution
            times.append(time.time() - start)
    return np.mean(times)

sizes = [500, 1000, 2000, 4000, 8000]
cpu_times = []
gpu_times = []

print("Benchmark Perkalian Matriks (CPU vs GPU):")
print(f"{'Ukuran':>8} {'CPU (ms)':>10} {'GPU (ms)':>10} {'Speedup':>10}")
print("-" * 42)

for size in sizes:
    cpu_t = benchmark_matmul(size, '/CPU:0')
    cpu_times.append(cpu_t * 1000)

    if tf.config.list_physical_devices('GPU'):
        gpu_t = benchmark_matmul(size, '/GPU:0')
        gpu_times.append(gpu_t * 1000)
        speedup = cpu_t / gpu_t
        print(f"{size:>8} {cpu_t*1000:>10.1f} {gpu_t*1000:>10.1f} {speedup:>9.1f}x")
    else:
        print(f"{size:>8} {cpu_t*1000:>10.1f} {'N/A':>10} {'N/A':>10}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

x_pos = np.arange(len(sizes))
width = 0.35

ax1.bar(x_pos - width/2, cpu_times, width, label='CPU', color='#42A5F5')
if gpu_times:
    ax1.bar(x_pos + width/2, gpu_times, width, label='GPU NVIDIA', color='#76B900')
ax1.set_xlabel('Ukuran Matriks')
ax1.set_ylabel('Waktu (ms) — log scale')
ax1.set_title('Waktu Perkalian Matriks', fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels([f'{s}x{s}' for s in sizes], rotation=45)
ax1.set_yscale('log')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

if gpu_times:
    speedups = [c/g for c, g in zip(cpu_times, gpu_times)]
    ax2.bar(x_pos, speedups, color='#76B900', edgecolor='white', linewidth=2)
    for i, s in enumerate(speedups):
        ax2.text(i, s * 1.05, f'{s:.1f}x', ha='center', fontweight='bold', fontsize=11)
    ax2.set_xlabel('Ukuran Matriks')
    ax2.set_ylabel('Speedup (x lipat)')
    ax2.set_title('GPU Speedup vs CPU', fontweight='bold')
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels([f'{s}x{s}' for s in sizes], rotation=45)
    ax2.grid(axis='y', alpha=0.3)
    ax2.set_ylim(0, max(speedups) * 1.3)

plt.suptitle('Benchmark: CPU vs GPU NVIDIA', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

if gpu_times:
    print(f"\nPada matriks besar ({sizes[-1]}x{sizes[-1]}), GPU {speedups[-1]:.0f}x lebih cepat!")

### Training Neural Network (Jaringan Saraf Tiruan): CPU vs GPU

Sekarang kita benchmark yang paling penting — training neural network sesungguhnya menggunakan dataset gambar CIFAR-10.

In [ ]:
# Load CIFAR-10
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0
# Use subset for faster benchmark
X_bench = X_train[:10000]
y_bench = y_train[:10000]
print(f"Data benchmark: {X_bench.shape[0]:,} gambar CIFAR-10")
print(f"Ukuran gambar : {X_bench.shape[1:]}")
print(f"Jumlah kelas  : 10 (pesawat, mobil, burung, kucing, dll)")

In [ ]:
def build_cnn():
    """Buat model CNN (Convolutional Neural Network) untuk benchmark"""
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(32, 32, 3)),
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])

# Benchmark CPU
print("Training CNN pada CPU...")
with tf.device('/CPU:0'):
    model_cpu = build_cnn()
    model_cpu.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    start = time.time()
    model_cpu.fit(X_bench, y_bench, epochs=3, batch_size=64, verbose=1)
    cpu_train_time = time.time() - start
print(f"CPU training time: {cpu_train_time:.1f} detik\n")

# Benchmark GPU
if tf.config.list_physical_devices('GPU'):
    print("Training CNN pada GPU NVIDIA...")
    with tf.device('/GPU:0'):
        model_gpu = build_cnn()
        model_gpu.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        # Warmup
        model_gpu.fit(X_bench[:64], y_bench[:64], epochs=1, verbose=0)
        start = time.time()
        model_gpu.fit(X_bench, y_bench, epochs=3, batch_size=64, verbose=1)
        gpu_train_time = time.time() - start
    print(f"GPU training time: {gpu_train_time:.1f} detik")
    print(f"\nGPU {cpu_train_time/gpu_train_time:.1f}x lebih cepat untuk training CNN!")
else:
    gpu_train_time = None
    print("GPU tidak tersedia. Aktifkan T4 GPU untuk melihat perbandingan.")

In [ ]:
if gpu_train_time is not None:
    fig, ax = plt.subplots(figsize=(8, 5))

    devices = ['CPU', 'GPU NVIDIA\n(T4)']
    times = [cpu_train_time, gpu_train_time]
    colors = ['#42A5F5', '#76B900']

    bars = ax.bar(devices, times, color=colors, edgecolor='white', linewidth=2, width=0.5)
    for bar, t in zip(bars, times):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.05,
                f'{t:.1f}s', ha='center', fontweight='bold', fontsize=14)

    ax.set_ylabel('Waktu Training (detik)')
    ax.set_title(
        f'Training CNN (3 epoch, 10K gambar) — GPU {cpu_train_time/gpu_train_time:.1f}x Lebih Cepat!',
        fontweight='bold', fontsize=13
    )
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, max(times) * 1.3)
    plt.tight_layout()
    plt.show()
else:
    print("Grafik tidak ditampilkan karena GPU tidak tersedia.")

### Inference (Prediksi): CPU vs GPU

Selain training, kecepatan **inference** (prediksi) juga sangat penting — terutama untuk aplikasi real-time seperti:
- Self-driving car (mobil otonom)
- Face recognition (pengenalan wajah)
- Chatbot dan asisten AI
- Deteksi objek di video

In [ ]:
# Benchmark inference
n_images = 1000
test_data = X_test[:n_images]

# CPU inference
print(f"Inference {n_images} gambar:")
with tf.device('/CPU:0'):
    model_inf = build_cnn()
    model_inf.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
    model_inf.fit(X_bench, y_bench, epochs=1, verbose=0)  # quick train

    start = time.time()
    _ = model_inf.predict(test_data, verbose=0)
    cpu_inf_time = time.time() - start
    print(f"  CPU: {cpu_inf_time*1000:.1f} ms ({n_images/cpu_inf_time:.0f} gambar/detik)")

if tf.config.list_physical_devices('GPU'):
    with tf.device('/GPU:0'):
        model_inf_gpu = build_cnn()
        model_inf_gpu.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
        model_inf_gpu.fit(X_bench, y_bench, epochs=1, verbose=0)
        _ = model_inf_gpu.predict(test_data[:10], verbose=0)  # warmup

        start = time.time()
        _ = model_inf_gpu.predict(test_data, verbose=0)
        gpu_inf_time = time.time() - start
        print(f"  GPU: {gpu_inf_time*1000:.1f} ms ({n_images/gpu_inf_time:.0f} gambar/detik)")
        print(f"\n  GPU {cpu_inf_time/gpu_inf_time:.1f}x lebih cepat untuk inference!")
else:
    gpu_inf_time = None
    print("  GPU tidak tersedia.")

In [ ]:
if gpu_inf_time is not None and gpu_train_time is not None:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    benchmarks = [
        ('Perkalian Matriks\n(8000x8000)', cpu_times[-1], gpu_times[-1], 'ms'),
        ('CNN Training\n(3 epoch)', cpu_train_time, gpu_train_time, 's'),
        ('CNN Inference\n(1000 gambar)', cpu_inf_time*1000, gpu_inf_time*1000, 'ms'),
    ]

    for ax, (title, cpu_val, gpu_val, unit) in zip(axes, benchmarks):
        bars = ax.bar(['CPU', 'GPU'], [cpu_val, gpu_val], color=['#42A5F5', '#76B900'],
                      edgecolor='white', linewidth=2)
        speedup = cpu_val / gpu_val
        ax.set_title(f'{title}\nGPU {speedup:.1f}x lebih cepat', fontweight='bold', fontsize=11)
        ax.set_ylabel(f'Waktu ({unit})')
        for bar, val in zip(bars, [cpu_val, gpu_val]):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.05,
                    f'{val:.1f}{unit}', ha='center', fontweight='bold', fontsize=10)
        ax.set_ylim(0, max(cpu_val, gpu_val) * 1.3)
        ax.grid(axis='y', alpha=0.3)

    plt.suptitle('Ringkasan Benchmark: CPU vs GPU NVIDIA', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("Grafik ringkasan tidak ditampilkan karena GPU tidak tersedia.")

## Bagian 3: Ekosistem NVIDIA untuk Deep Learning

NVIDIA bukan hanya pembuat GPU — mereka menyediakan **ekosistem lengkap** (complete ecosystem) untuk deep learning:

### Software Stack (Tumpukan Perangkat Lunak)

| Layer | Komponen | Fungsi |
|-------|---------|--------|
| **Hardware** | GPU (T4, A100, H100) | Chip yang menjalankan komputasi |
| **Driver** | NVIDIA Driver | Komunikasi OS ke GPU |
| **Runtime** | CUDA | Platform komputasi paralel |
| **Library** | cuDNN, cuBLAS, TensorRT | Optimasi operasi deep learning |
| **Framework** | TensorFlow, PyTorch | Framework yang kita gunakan |
| **Tools** | NGC, NeMo, Triton | Container, model, serving |

### GPU NVIDIA untuk Berbagai Kebutuhan

| GPU | Target | Memory | Harga (est.) |
|-----|--------|--------|-------------|
| **T4** | Cloud inference, Colab | 16 GB | $2,000 |
| **RTX 4090** | Desktop/workstation | 24 GB | $1,600 |
| **A100** | Data center training | 80 GB | $10,000 |
| **H100** | AI supercomputer | 80 GB | $25,000 |
| **Jetson Orin** | Edge/robotics | 8–64 GB | $200–2,000 |

> Di Google Colab, kamu menggunakan **NVIDIA Tesla T4** — GPU yang sama digunakan oleh perusahaan untuk AI inference!

In [ ]:
print("=" * 60)
print("    NVIDIA Deep Learning Software Stack")
print("=" * 60)
print()

stack = [
    ("Tools & Apps", "NGC Catalog | NeMo | Triton Inference Server"),
    ("Frameworks  ", "TensorFlow | PyTorch | JAX | MXNet"),
    ("Libraries   ", "cuDNN | cuBLAS | NCCL | TensorRT"),
    ("Runtime     ", "CUDA Toolkit"),
    ("Driver      ", "NVIDIA GPU Driver"),
    ("Hardware    ", "GPU (T4 / A100 / H100 / Jetson)"),
]

for name, components in stack:
    print(f"  [{name}]  {components}")
    print(f"  {'─' * 50}")

print()
if tf.config.list_physical_devices('GPU'):
    print("Status di Colab ini:")
    print(f"  CUDA built   : {tf.test.is_built_with_cuda()}")
    print(f"  GPU available: {tf.config.list_physical_devices('GPU')}")
else:
    print("GPU belum diaktifkan — aktifkan T4 GPU untuk hasil optimal.")

## Bagian 4: Mixed Precision Training

GPU NVIDIA modern memiliki **Tensor Cores** yang sangat cepat untuk operasi float16. **Mixed Precision** menggabungkan float16 (cepat) dan float32 (akurat) untuk training yang lebih cepat tanpa mengorbankan akurasi.

| Tipe Data | Bit | Kecepatan | Akurasi |
|-----------|-----|-----------|--------|
| **Float32** | 32 bit per angka | Normal | Tinggi |
| **Float16** | 16 bit per angka | 2x lebih cepat | Sedikit lebih rendah |
| **Mixed Precision** | Keduanya | Mendekati float16 | Mendekati float32 |

Strategi Mixed Precision:
- Gunakan **float16** untuk komputasi maju dan mundur (forward & backward pass)
- Gunakan **float32** untuk akumulasi gradien (gradient accumulation) agar numerik stabil

Model besar seperti GPT-4, Stable Diffusion, dan Gemini semuanya dilatih dengan Mixed Precision di GPU NVIDIA!

In [ ]:
from tensorflow.keras import mixed_precision

# Aktifkan Mixed Precision
mixed_precision.set_global_policy('mixed_float16')
print(f"Global policy: {mixed_precision.global_policy().name}")

# Build model dengan mixed precision
# Output layer harus float32 untuk kestabilan numerik
mp_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(32, 32, 3)),
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax', dtype='float32')  # float32 untuk output
])
mp_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("\nTraining dengan Mixed Precision...")
start = time.time()
mp_model.fit(X_bench, y_bench, epochs=3, batch_size=64, verbose=1)
mp_time = time.time() - start
print(f"Mixed Precision time: {mp_time:.1f}s")

# Reset policy ke default
mixed_precision.set_global_policy('float32')
print(f"Policy direset ke: {mixed_precision.global_policy().name}")

print(f"\nPerbandingan training 3 epoch:")
print(f"  CPU (float32)           : {cpu_train_time:.1f}s")
if gpu_train_time is not None:
    print(f"  GPU (float32)           : {gpu_train_time:.1f}s")
    print(f"  GPU (mixed precision)   : {mp_time:.1f}s")
    if mp_time < gpu_train_time:
        print(f"  Mixed Precision {gpu_train_time/mp_time:.1f}x lebih cepat dari GPU float32!")
else:
    print(f"  Mixed Precision         : {mp_time:.1f}s")

Mixed Precision Training adalah teknik standar industri. Model besar seperti GPT-4, Stable Diffusion, dan Gemini semua dilatih dengan mixed precision di GPU NVIDIA.

Keuntungan Mixed Precision:
- Training lebih cepat (memanfaatkan Tensor Cores)
- Penggunaan memori GPU lebih efisien (hemat ~50%)
- Akurasi tidak berkurang karena gradien tetap di float32

## Bagian 5: NVIDIA TensorRT — Optimasi Inference

Setelah model dilatih, **TensorRT** mengoptimasi model untuk inference yang lebih cepat:
- **Layer fusion**: Menggabungkan beberapa layer menjadi satu operasi (misal: Conv + BN + ReLU → 1 op)
- **Precision calibration**: Otomatis memilih float16/int8 per layer
- **Kernel auto-tuning**: Memilih algoritma tercepat untuk hardware spesifik

TensorRT bisa mempercepat inference **2-5x** di atas GPU biasa!

Di Google Colab, kita bisa menggunakan **TF-TRT** (TensorFlow-TensorRT) yang sudah terintegrasi.
Alur kerjanya:
1. Train model seperti biasa
2. Simpan model (SavedModel format)
3. Konversi dengan TF-TRT
4. Benchmark: model asli vs model TensorRT

In [ ]:
# === Praktik TensorRT dengan TF-TRT ===

# Langkah 1: Simpan model sebagai SavedModel
saved_model_dir = 'saved_model_cifar10'
model_cpu.save(saved_model_dir)
print(f"Model disimpan ke: {saved_model_dir}/")

if tf.config.list_physical_devices('GPU'):
    try:
        from tensorflow.python.compiler.tensorrt import trt_convert as trt
        print("TF-TRT tersedia! Melakukan konversi...\n")

        # Langkah 2: Konversi ke TensorRT (FP16)
        converter = trt.TrtGraphConverterV2(
            input_saved_model_dir=saved_model_dir,
            conversion_params=trt.TrtConversionParams(
                precision_mode=trt.TrtPrecisionMode.FP16,
                max_workspace_size_bytes=1 << 30  # 1 GB
            )
        )
        converter.convert()

        # Simpan model TensorRT
        trt_model_dir = 'trt_model_cifar10_fp16'
        converter.save(trt_model_dir)
        print(f"Model TensorRT (FP16) disimpan ke: {trt_model_dir}/")

        # Langkah 3: Benchmark — Model Asli vs TensorRT
        print("\nBenchmark Inference (1000 gambar):")
        print("-" * 45)

        benchmark_data = X_test[:1000]

        # Model asli
        original_model = tf.saved_model.load(saved_model_dir)
        infer_original = original_model.signatures['serving_default']
        input_name = list(infer_original.structured_input_signature[1].keys())[0]

        # Warmup
        _ = infer_original(**{input_name: tf.constant(benchmark_data[:10])})
        start = time.time()
        for i in range(0, 1000, 100):
            _ = infer_original(**{input_name: tf.constant(benchmark_data[i:i+100])})
        original_time = time.time() - start
        print(f"  Model asli (FP32) : {original_time*1000:.1f} ms")

        # Model TensorRT
        trt_model = tf.saved_model.load(trt_model_dir)
        infer_trt = trt_model.signatures['serving_default']

        # Warmup (TRT builds engine on first call)
        _ = infer_trt(**{input_name: tf.constant(benchmark_data[:10])})
        start = time.time()
        for i in range(0, 1000, 100):
            _ = infer_trt(**{input_name: tf.constant(benchmark_data[i:i+100])})
        trt_time = time.time() - start
        print(f"  TensorRT (FP16)   : {trt_time*1000:.1f} ms")
        print(f"  Speedup           : {original_time/trt_time:.2f}x")

        # Ukuran model
        import subprocess
        orig_size = subprocess.check_output(['du', '-sh', saved_model_dir]).decode().split()[0]
        trt_size = subprocess.check_output(['du', '-sh', trt_model_dir]).decode().split()[0]
        print(f"\n  Ukuran model asli   : {orig_size}")
        print(f"  Ukuran model TRT    : {trt_size}")

    except ImportError:
        print("TF-TRT tidak tersedia di environment ini.")
        print("Pastikan menggunakan GPU runtime di Google Colab.")
    except Exception as e:
        print(f"TensorRT conversion error: {e}")
        print("Ini bisa terjadi jika versi TF/TRT tidak kompatibel.")
        print("Konsep tetap sama: TensorRT mengoptimasi inference 2-5x lebih cepat.")
else:
    print("TensorRT memerlukan GPU NVIDIA.")
    print("Aktifkan GPU di Colab: Runtime -> Change runtime type -> T4 GPU")
    print("\nAlur kerja TensorRT:")
    print("  1. Train model (TensorFlow/PyTorch)")
    print("  2. Export model (SavedModel)")
    print("  3. Konversi dengan TF-TRT (FP16/INT8)")
    print("  4. Deploy model yang teroptimasi")

In [ ]:
# Visualisasi perbandingan TensorRT (jika berhasil)
if tf.config.list_physical_devices('GPU'):
    try:
        # Cek apakah variabel benchmark ada
        _ = original_time, trt_time

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

        # Bar chart waktu
        labels = ['Model Asli\n(FP32)', 'TensorRT\n(FP16)']
        times_ms = [original_time * 1000, trt_time * 1000]
        colors = ['#42A5F5', '#76B900']

        bars = ax1.bar(labels, times_ms, color=colors, edgecolor='white', linewidth=2)
        for bar, t in zip(bars, times_ms):
            ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.05,
                     f'{t:.1f} ms', ha='center', fontweight='bold', fontsize=12)
        ax1.set_ylabel('Waktu Inference (ms)')
        ax1.set_title('Inference: 1000 Gambar', fontweight='bold')
        ax1.grid(axis='y', alpha=0.3)
        ax1.set_ylim(0, max(times_ms) * 1.4)

        # Throughput
        throughputs = [1000 / original_time, 1000 / trt_time]
        bars2 = ax2.bar(labels, throughputs, color=colors, edgecolor='white', linewidth=2)
        for bar, tp in zip(bars2, throughputs):
            ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.05,
                     f'{tp:.0f} img/s', ha='center', fontweight='bold', fontsize=12)
        ax2.set_ylabel('Throughput (gambar/detik)')
        ax2.set_title('Throughput Inference', fontweight='bold')
        ax2.grid(axis='y', alpha=0.3)
        ax2.set_ylim(0, max(throughputs) * 1.4)

        speedup = original_time / trt_time
        plt.suptitle(f'NVIDIA TensorRT: {speedup:.2f}x Lebih Cepat!',
                     fontsize=15, fontweight='bold')
        plt.tight_layout()
        plt.show()
    except NameError:
        print("Benchmark TensorRT belum dijalankan.")

## Ringkasan (Summary)

Selamat! Kamu telah menyelesaikan notebook **Akselerasi GPU NVIDIA untuk Deep Learning**.

### Apa yang Sudah Kita Pelajari

| Topik | Poin Utama |
|-------|------------|
| **GPU vs CPU** | GPU punya ribuan core kecil untuk parallel processing, CPU punya sedikit core kuat untuk serial |
| **Perkalian Matriks** | GPU **10–100x** lebih cepat untuk matriks besar |
| **CNN Training** | GPU mempercepat training neural network secara signifikan |
| **Inference** | GPU bisa memproses ratusan hingga ribuan gambar per detik |
| **NVIDIA Stack** | CUDA → cuDNN → Framework → TensorRT (dari hardware ke aplikasi) |
| **Mixed Precision** | Gabungan float16 + float32 untuk training lebih cepat dan efisien |
| **TensorRT** | Optimasi model untuk inference 2–5x lebih cepat di GPU NVIDIA |

### Benchmark yang Kita Lakukan

Kita telah mengukur tiga benchmark utama:
1. **Perkalian Matriks** — operasi dasar deep learning
2. **CNN Training** — melatih model klasifikasi gambar CIFAR-10
3. **CNN Inference** — prediksi batch gambar

Hasilnya: GPU NVIDIA secara konsisten lebih cepat, terutama untuk data berukuran besar.

### Ekosistem NVIDIA

```
[Tools & Apps]  NGC | NeMo | Triton Inference Server
[Frameworks  ]  TensorFlow | PyTorch | JAX
[Libraries   ]  cuDNN | cuBLAS | NCCL | TensorRT
[Runtime     ]  CUDA Toolkit
[Driver      ]  NVIDIA GPU Driver
[Hardware    ]  T4 / A100 / H100 / Jetson
```

---

### Selanjutnya

**Modul 03 — NLP Fundamentals (Natural Language Processing)**

Kita akan belajar bagaimana komputer memahami bahasa manusia: tokenisasi, word embedding, dan model bahasa. GPU yang sudah kita pelajari hari ini akan terus kita gunakan di modul-modul berikutnya!